# Implementasi Bab 4 — Feature Engineering dan Pipeline

Feature construction, ColumnTransformer, encoding, feature selection, dan Pipeline end-to-end [ML-MODUL-01, hlm. 32–34]. Dataset properti dibuat sintetis agar notebook mandiri di Colab.


In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
rng = np.random.default_rng(SEED)
n_samples = 500
properties = pd.DataFrame({
    "luas_bangunan": rng.normal(120, 40, n_samples).clip(30),
    "jumlah_kamar": rng.integers(1, 6, n_samples),
    "tipe_properti": rng.choice(["Rumah", "Apartemen", "Ruko"], n_samples),
    "kondisi_bangunan": rng.choice(["Buruk", "Sedang", "Baik", "Sangat Baik"], n_samples),
})
properties["harga"] = (
    properties["luas_bangunan"] * 15
    + properties["jumlah_kamar"] * 20
    + rng.normal(0, 50, n_samples)
) * 1_000_000
properties["rasio_kamar_per_luas"] = properties["jumlah_kamar"] / properties["luas_bangunan"]
display(properties.head())


In [ ]:
numeric_features = ["luas_bangunan", "jumlah_kamar", "rasio_kamar_per_luas"]
categorical_features = ["tipe_properti"]
ordinal_features = ["kondisi_bangunan"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
    ("ord", OrdinalEncoder(categories=[["Buruk", "Sedang", "Baik", "Sangat Baik"]]), ordinal_features),
])


In [ ]:
X = properties.drop(columns="harga")
y = properties["harga"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
property_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1)),
])
property_pipeline.fit(X_train, y_train)
y_pred = property_pipeline.predict(X_test)
print("R2:", round(r2_score(y_test, y_pred), 3))


In [ ]:
plt.scatter(y_test, y_pred, alpha=0.6)
limits = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(limits, limits, "r--")
plt.xlabel("Harga Aktual")
plt.ylabel("Harga Prediksi")
plt.title("Prediksi vs Aktual — Pipeline Lengkap")
plt.show()


## Latihan: tiga pendekatan feature selection

Contoh berikut menggunakan dataset klasifikasi bawaan, sehingga tidak memerlukan unduhan Titanic. Filter diwakili SelectKBest, wrapper oleh RFE, dan embedded oleh koefisien Logistic Regression berpenalti L1 [ML-MODUL-01, hlm. 34].


In [ ]:
cancer = load_breast_cancer(as_frame=True)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=SEED, stratify=cancer.target
)

filter_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(score_func=f_classif, k=10)),
    ("model", LogisticRegression(max_iter=2000, random_state=SEED)),
])
filter_pipeline.fit(X_train_c, y_train_c)

wrapper_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", RFE(LogisticRegression(max_iter=2000, random_state=SEED), n_features_to_select=10)),
    ("model", LogisticRegression(max_iter=2000, random_state=SEED)),
])
wrapper_pipeline.fit(X_train_c, y_train_c)

embedded_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(penalty="l1", solver="liblinear", max_iter=2000, random_state=SEED)),
])
embedded_pipeline.fit(X_train_c, y_train_c)

filter_features = list(cancer.feature_names[filter_pipeline.named_steps["selector"].get_support()])
rfe_features = list(cancer.feature_names[wrapper_pipeline.named_steps["selector"].get_support()])
embedded_mask = np.abs(embedded_pipeline.named_steps["model"].coef_).max(axis=0) > 1e-8
embedded_features = list(cancer.feature_names[embedded_mask])
selected_feature_names = {
    "SelectKBest": filter_features,
    "RFE": rfe_features,
    "L1 embedded": embedded_features,
}

for name, candidate in {
    "SelectKBest": filter_pipeline,
    "RFE": wrapper_pipeline,
    "L1 embedded": embedded_pipeline,
}.items():
    print(name, round(accuracy_score(y_test_c, candidate.predict(X_test_c)), 3))
    print("Fitur terpilih:", selected_feature_names[name])


In [ ]:
new_property = pd.DataFrame([{
    "luas_bangunan": 100,
    "jumlah_kamar": 3,
    "tipe_properti": "Rumah",
    "kondisi_bangunan": "Baik",
    "rasio_kamar_per_luas": 3 / 100,
}])
print("Prediksi harga data baru:", f"Rp{property_pipeline.predict(new_property)[0]:,.0f}")


## Kesimpulan

Pipeline menyatukan preprocessing dan model sehingga fitting pada train serta prediksi data baru konsisten. Ini mengurangi risiko leakage dan kesalahan urutan transformasi [ML-MODUL-01, hlm. 34].
